# Imports

In [2]:
import os
import sys
import time
import gc
import json
import pickle
import numpy as np
import pandas as pd
import psutil
import requests

from PIL import Image
import torch
import torch.nn as nn
from tqdm.notebook import tqdm

from torchvision import models, transforms
from transformers import AutoModel, AutoImageProcessor


from transformers import (
    ViTModel, ViTImageProcessor,
    DeiTModel, DeiTImageProcessor,
    CLIPProcessor, CLIPVisionModel,
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    GPT2Tokenizer, GPT2Model,
    CLIPTextModel
)

/home/aysel/tfe/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from paths import EMBED_DIR, vision_emb_path, text_emb_path, ensure_dirs, vision_artifact_dir, text_artifact_dir
ensure_dirs()

# Configuration

In [4]:
# Dataset selection
CURRENT_DATASET = "Flickr8k"
ALL_DATASETS = ["Flickr8k", "Flickr30k", "ConceptualCaptions"]

BASE_DIR = os.path.join(os.getcwd(), "TFE_Data")
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")
RESULTS_DIR = os.path.join(BASE_DIR, "01_Unimodal_Embeddings", CURRENT_DATASET)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

Using device: cuda


In [5]:
df_path = os.path.join(DATASETS_DIR, f"df_{CURRENT_DATASET}.pkl")
df = pd.read_pickle(df_path)

IMAGE_PATHS = df["image_path"].tolist()
CAPTIONS_LIST = df["captions"].tolist()

# Keep for future XAI (but not used in indexation)
texts_for_xai = [caps[0] for caps in CAPTIONS_LIST]
caption_image_map = {i: IMAGE_PATHS[i] for i in range(len(IMAGE_PATHS))}

# Flatten all captions for text extraction
flattened_captions = [cap for caps in CAPTIONS_LIST for cap in caps]

print(f"Loaded {len(IMAGE_PATHS)} images and {len(flattened_captions)} captions.")


Loaded 8091 images and 40455 captions.


# Utility Functions

## GreenAI Metrics

In [6]:
def measure_memory():
    """Returns the current memory usage of the process in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

In [7]:
def get_size_in_mb(obj):
    """Returns the size of a numpy array in MB."""
    if isinstance(obj, np.ndarray):
        return obj.nbytes / (1024 * 1024)
    else:
        return sys.getsizeof(obj) / (1024 * 1024)

In [8]:
"""def save_metadata(data, dataset, modality, model_name):
    model_dir = os.path.join(BASE_DIR, "Unimodal_Results", dataset, modality, model_name)
    os.makedirs(model_dir, exist_ok=True)
    meta_path = os.path.join(model_dir, "metadata.pkl")
    with open(meta_path, "wb") as f:
        pickle.dump({"samples": data, "indices": list(range(len(data)))}, f)
        """
def save_metadata(data, dataset, modality, model_name):
    if modality == "vision":
        model_dir = vision_artifact_dir(dataset, model_name)
    else:
        model_dir = text_artifact_dir(dataset, model_name)

    os.makedirs(model_dir, exist_ok=True)
    meta_path = os.path.join(model_dir, "metadata.pkl")

    with open(meta_path, "wb") as f:
        pickle.dump({"samples": data, "indices": list(range(len(data)))}, f)


In [9]:
def execute_and_save(dataset_name, modality, model_name, extract_func, data, device):
    start = time.time()
    mem_before = measure_memory()

    try:
        extract_func(data, device, dataset_name)
    except Exception as e:
        print(f"[ERROR] {model_name} failed: {e}")
        return None

    exec_time = time.time() - start
    mem_used = max(0, measure_memory() - mem_before)

    return {
        "Dataset": dataset_name,
        "Modality": modality,
        "Model": model_name,
        "Num_Samples": len(data),
        "Time_s": exec_time,
        "Latency_s": exec_time / len(data),
        "Throughput_samples_per_s": len(data) / exec_time,
        "Memory_MB": mem_used
    }

# Unimodal Models

## CBIR: Vision Feature Extractions

### Image Dataset

In [10]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = [p for p in image_paths if os.path.exists(p)]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            img = Image.open(path).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            try:
                img = self.transform(img)
            except:
                img = torch.zeros((3, 224, 224))

        return img


### Feature Extraction

In [11]:
def vision_extract_features(model, model_name, dataloader, device, target_layer, image_paths, dataset_name):

    model.eval()

    features_list = []
    hidden_states_list = []
    attn_maps_list = []
    cls_tokens_list = []
    logits_list = []

    """model_dir = os.path.join(BASE_DIR, "Unimodal_Results", dataset_name, "vision", model_name)
    os.makedirs(model_dir, exist_ok=True)"""
    
    model_dir = vision_artifact_dir("Flickr8k", model_name)
    os.makedirs(model_dir, exist_ok=True)

    # Hidden states hook
    def hook_hidden(module, inp, out):
        if hasattr(out, "last_hidden_state"):
            hs = out.last_hidden_state.detach().cpu()
        elif isinstance(out, tuple):
            hs = out[0].detach().cpu()
        else:
            hs = out.detach().cpu()
        hidden_states_list.append(hs)

    hook_handle = target_layer.register_forward_hook(hook_hidden)

    # Attention maps (Transformers only)
    attn_handle = None
    try:
        attn_layer = model.encoder.layer[-1].attention.attention
        def hook_attn(module, inp, out):
            attn_maps_list.append(out[0].detach().cpu())
        attn_handle = attn_layer.register_forward_hook(hook_attn)
    except:
        pass

    # Main loop
    for batch_idx, batch in enumerate(tqdm(dataloader, desc=model_name)):
        batch = batch.to(device)
        print(f"Processed batch {batch_idx} | shape: {batch.shape}")
        torch.save(batch.cpu(), os.path.join(model_dir, f"pixel_values_{batch_idx}.pt"))

        out = model(batch)

        if hasattr(out, "last_hidden_state"):
            cls_tokens_list.append(out.last_hidden_state[:, 0, :].detach().cpu())
            feat = out.last_hidden_state[:, 0, :]
        else:
            logits_list.append(out.detach().cpu())
            if out.ndim == 4:
                feat = torch.nn.functional.adaptive_avg_pool2d(out, 1).view(out.size(0), -1)
            else:
                feat = out
        
        # Save patch embeddings (all tokens except CLS)
        if hasattr(out, "last_hidden_state"):
            patch_tokens = out.last_hidden_state[:, 1:, :].detach().cpu()
            torch.save(patch_tokens, os.path.join(model_dir, f"patch_tokens_{batch_idx}.pt"))

        if attn_maps_list:
            torch.save(attn_maps_list[-1], os.path.join(model_dir, f"attn_{batch_idx}.pt"))
        features_list.append(feat.detach().cpu().numpy())

    hook_handle.remove()
    if attn_handle:
        attn_handle.remove()

    # Save outputs
    save_path = vision_emb_path(dataset_name, model_name)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    np.save(save_path, np.vstack(features_list))

    # OLD SAVE PATH BEFORE CLEANUP: np.save(os.path.join(model_dir, "embeddings.npy"), np.vstack(features_list))

    # Metadata
    model_info = {
        "model_name": model_name,
        "architecture": model.__class__.__name__,
        "embedding_dim": features_list[0].shape[-1],
        "num_parameters": sum(p.numel() for p in model.parameters()),
        "device": str(device)
    }

    with open(os.path.join(model_dir, "model_info.json"), "w") as f:
        json.dump(model_info, f, indent=4)


#### CNN Embeddings

In [12]:
def get_resnet50_embeddings(image_paths, device, dataset_name):
    try:
        model = models.resnet50(weights="DEFAULT").to(device)
        target_layer = model.layer4[-1]
        model.fc = nn.Identity()  # Remove classifier

        transform = models.ResNet50_Weights.DEFAULT.transforms()
        dataset = ImageDataset(image_paths, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)

        return vision_extract_features(model, "resnet50", loader, device, target_layer, image_paths, dataset_name)

    except Exception as e:
        print(f"[ERROR] ResNet50 embedding failed: {e}")
        return None

In [13]:
def get_mobilenet_v3_embeddings(image_paths, device, dataset_name):
    try:
        model = models.mobilenet_v3_large(weights="DEFAULT").to(device)
        target_layer = model.features[-1]
        model.classifier = nn.Identity()  # Remove classifier

        transform = models.MobileNet_V3_Large_Weights.DEFAULT.transforms()
        dataset = ImageDataset(image_paths, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4)

        return vision_extract_features(model, "mobilenet_v3", loader, device,
                                              target_layer, image_paths, dataset_name)
    except Exception as e:
        print(f"[ERROR] MobileNetV3 embedding failed: {e}")
        return None

#### Transformer Embeddings

In [14]:
def get_vit_embeddings(image_paths, device, dataset_name):
    try:
        processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
        model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k').to(device)

        target_layer = model.encoder.layer[-1]

        def collate_fn(batch):
            processed_imgs = []
            for img in batch:
                # Ensure image is a PIL Image
                try:
                    if not isinstance(img, Image.Image):
                        raise ValueError("Invalid image type")
                    img = img.convert('RGB')  # force RGB
                    pixel_values = processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
                except Exception as e:
                    # Fallback black RGB image if something fails
                    print(f"[WARNING] Replacing corrupted image with fallback | {e}")
                    fallback = Image.new('RGB', (224, 224), (0, 0, 0))
                    pixel_values = processor(images=fallback, return_tensors="pt")['pixel_values'].squeeze(0)
                processed_imgs.append(pixel_values)
            return torch.stack(processed_imgs)

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=32, num_workers=4, collate_fn=collate_fn
        )

        return vision_extract_features(
            model, "vit", loader, device,
            target_layer, image_paths, dataset_name,
            
        )

    except Exception as e:
        print(f"[ERROR] ViT embedding failed: {e}")
        return None

In [15]:
def get_pvt_embeddings(image_paths, device, dataset_name):
    try:
        # Load model + processor
        model = AutoModel.from_pretrained("Zetatech/pvt-tiny-224").to(device)
        processor = AutoImageProcessor.from_pretrained("Zetatech/pvt-tiny-224")

        # Correct deepest transformer block
        target_layer = model.encoder.block[3][1]

        # Collate function
        def collate_fn(batch):
            processed = []
            for img in batch:
                try:
                    if not isinstance(img, Image.Image):
                        raise ValueError("Invalid image type")
                    img = img.convert("RGB")
                    pixel_values = processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
                except:
                    fallback = Image.new("RGB", (224, 224), (0, 0, 0))
                    pixel_values = processor(images=fallback, return_tensors="pt")["pixel_values"].squeeze(0)
                processed.append(pixel_values)
            return torch.stack(processed)

        dataset = ImageDataset(image_paths)
        loader = torch.utils.data.DataLoader(dataset, batch_size=32, num_workers=4, collate_fn=collate_fn)

        # Extract features (this saves embeddings.npy)
        return vision_extract_features(
            model, "pvt", loader, device,
            target_layer, image_paths, dataset_name
        )

    except Exception as e:
        print(f"[ERROR] PVT embedding failed: {e}")
        return None


## T2T: Text Feature Extractions


### Text Dataset

In [16]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}


### Text Feature Extraction

In [17]:
def text_extract_features(model, tokenizer, dataloader, device, dataset_name, model_name):

    model.eval()

    features_list = []
    hidden_states_list = []
    cls_tokens_list = []

    """model_dir = os.path.join(BASE_DIR, "Unimodal_Results", dataset_name, "text", model_name)
    os.makedirs(model_dir, exist_ok=True)"""
    model_dir = text_artifact_dir("Flickr8k", model_name)
    os.makedirs(model_dir, exist_ok=True)


    def hook_hidden(module, inp, out):
        hs = out.last_hidden_state.detach().cpu()
        hidden_states_list.append(hs)

    hook_handle = model.register_forward_hook(hook_hidden)

    for batch_idx, batch in enumerate(tqdm(dataloader, desc=f"{model_name} Text")):
        batch = {k: v.to(device) for k, v in batch.items()}

        torch.save(batch["input_ids"].cpu(), os.path.join(model_dir, f"input_ids_{batch_idx}.pt"))
        torch.save(batch["attention_mask"].cpu(), os.path.join(model_dir, f"attention_mask_{batch_idx}.pt"))

        with torch.no_grad():
            out = model(**batch)
            
            # Save token embeddings
            token_embeddings = out.last_hidden_state.detach().cpu()
            torch.save(token_embeddings, os.path.join(model_dir, f"token_embeddings_{batch_idx}.pt"))

        cls_tokens_list.append(out.last_hidden_state[:, 0, :].detach().cpu())
        torch.save(out.last_hidden_state[:, 0, :].detach().cpu(), os.path.join(model_dir, f"cls_tokens_{batch_idx}.pt"))

        mask = batch["attention_mask"].unsqueeze(-1).float()
        sum_emb = torch.sum(out.last_hidden_state * mask, dim=1)
        sum_mask = torch.clamp(mask.sum(1), min=1e-9)
        features_list.append((sum_emb / sum_mask).detach().cpu().numpy())

    hook_handle.remove()
    
    save_path = text_emb_path(dataset_name, model_name)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    np.save(save_path, np.vstack(features_list))


    #OLD SAVE LINE BEFORE CLEANUP: np.save(os.path.join(model_dir, "embeddings.npy"), np.vstack(features_list))

    model_info = {
        "model_name": model_name,
        "architecture": model.__class__.__name__,
        "embedding_dim": features_list[0].shape[-1],
        "num_parameters": sum(p.numel() for p in model.parameters()),
        "device": str(device)
    }

    with open(os.path.join(model_dir, "model_info.json"), "w") as f:
        json.dump(model_info, f, indent=4)


### Text Models

In [18]:
def get_bert_embeddings(texts, device, dataset_name, caption_image_map=None):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "bert")


In [19]:

def get_roberta_embeddings(texts, device, dataset_name, caption_image_map=None):
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    model = RobertaModel.from_pretrained('roberta-base').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "roberta")


In [20]:

def get_gpt2_embeddings(texts, device, dataset_name, caption_image_map=None):
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2Model.from_pretrained('gpt2').to(device)
    dataset = TextDataset(texts, tokenizer)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)
    return text_extract_features(model, tokenizer, dataloader, device, dataset_name, "gpt2")


# Execution

In [21]:
all_metrics = []

vision_pipeline = {
    "ResNet50": get_resnet50_embeddings,
    "MobileNetV3": get_mobilenet_v3_embeddings,
    "ViT": get_vit_embeddings,
    "PVT": get_pvt_embeddings,
}

text_pipeline = {
    "BERT": get_bert_embeddings,
    "RoBERTa": get_roberta_embeddings,
    "GPT2": get_gpt2_embeddings,
}

for ds_name in ["Flickr8k"]:
    print(f"\n=== Starting {ds_name} ===")

    # Vision
    for name, func in vision_pipeline.items():
        m = execute_and_save(ds_name, "vision", name, func, IMAGE_PATHS, device)
        if m: all_metrics.append(m)

    # Text
    for name, func in text_pipeline.items():
        m = execute_and_save(ds_name, "text", name, func, flattened_captions, device)
        if m: all_metrics.append(m)

df_final = pd.DataFrame(all_metrics)
df_final.to_pickle(os.path.join(EMBED_DIR, "global_unimodal_metrics.pkl"))

print("Indexation complete.")



=== Starting Flickr8k ===


resnet50:   0%|          | 0/253 [00:00<?, ?it/s]

Processed batch 0 | shape: torch.Size([32, 3, 224, 224])
Processed batch 1 | shape: torch.Size([32, 3, 224, 224])
Processed batch 2 | shape: torch.Size([32, 3, 224, 224])
Processed batch 3 | shape: torch.Size([32, 3, 224, 224])
Processed batch 4 | shape: torch.Size([32, 3, 224, 224])
Processed batch 5 | shape: torch.Size([32, 3, 224, 224])
Processed batch 6 | shape: torch.Size([32, 3, 224, 224])
Processed batch 7 | shape: torch.Size([32, 3, 224, 224])
Processed batch 8 | shape: torch.Size([32, 3, 224, 224])
Processed batch 9 | shape: torch.Size([32, 3, 224, 224])
Processed batch 10 | shape: torch.Size([32, 3, 224, 224])
Processed batch 11 | shape: torch.Size([32, 3, 224, 224])
Processed batch 12 | shape: torch.Size([32, 3, 224, 224])
Processed batch 13 | shape: torch.Size([32, 3, 224, 224])
Processed batch 14 | shape: torch.Size([32, 3, 224, 224])
Processed batch 15 | shape: torch.Size([32, 3, 224, 224])
Processed batch 16 | shape: torch.Size([32, 3, 224, 224])
Processed batch 17 | sha

mobilenet_v3:   0%|          | 0/253 [00:00<?, ?it/s]

Processed batch 0 | shape: torch.Size([32, 3, 224, 224])
Processed batch 1 | shape: torch.Size([32, 3, 224, 224])
Processed batch 2 | shape: torch.Size([32, 3, 224, 224])
Processed batch 3 | shape: torch.Size([32, 3, 224, 224])
Processed batch 4 | shape: torch.Size([32, 3, 224, 224])
Processed batch 5 | shape: torch.Size([32, 3, 224, 224])
Processed batch 6 | shape: torch.Size([32, 3, 224, 224])
Processed batch 7 | shape: torch.Size([32, 3, 224, 224])
Processed batch 8 | shape: torch.Size([32, 3, 224, 224])
Processed batch 9 | shape: torch.Size([32, 3, 224, 224])
Processed batch 10 | shape: torch.Size([32, 3, 224, 224])
Processed batch 11 | shape: torch.Size([32, 3, 224, 224])
Processed batch 12 | shape: torch.Size([32, 3, 224, 224])
Processed batch 13 | shape: torch.Size([32, 3, 224, 224])
Processed batch 14 | shape: torch.Size([32, 3, 224, 224])
Processed batch 15 | shape: torch.Size([32, 3, 224, 224])
Processed batch 16 | shape: torch.Size([32, 3, 224, 224])
Processed batch 17 | sha

...

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


roberta Text:   0%|          | 0/1265 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

gpt2 Text:   0%|          | 0/1265 [00:00<?, ?it/s]

Indexation complete.
